# PII Detection and Redaction Demo

## Supported Formats
- **PDF** (.pdf) — text-based and image-based processing
- **Word** (.docx)
- **Tabular** (.xlsx, .csv, .JSON)
- **Text** (.txt)
- **Images** (.jpg, .jpeg, .png, .tiff, .tif, .bmp, .webp)

## 3-Step Pipeline
Both approaches follow the same pipeline:

| Step | Description |
| --- | --- |
| **Step 1 — PII Detection** | Identify PII entities (names, SSNs, addresses, etc.) using Amazon Bedrock |
| **Step 2 — Synthetic Generation** | Generate realistic replacement values via LLM (or Faker fallback) |
| **Step 3 — Redaction** | Replace original PII with synthetic data in the document |

## Two Approaches (PDF)
1. **Text-Based**: Extracts text → detects PII → replaces in text → outputs `.txt`
2. **Image-Based**: Converts pages to images → vision LLM detects PII → Textract refines bounding boxes → flattens each page to a redacted image → outputs `.pdf`

## Redaction Options (`src/config.yaml`)

| Option | Values | Description |
| --- | --- | --- |
| `redaction.mode` | `synthetic` (default), `blackout` | **synthetic**: replaces PII with realistic fake data. **blackout**: covers PII with solid black rectangles |
| `show_bounding_boxes` | `false` (default), `true` | When `true`, draws visible colored borders around redacted areas — useful for review and audit |
| `bounding_box_color` | `[0.8, 0.2, 0.2]` (soft red) | RGB color (0-1) for bounding box borders when `show_bounding_boxes` is enabled |

This notebook mirrors the Lambda implementation for local testing.


## Setup

### Prerequisites

**AWS Credentials Required**: This notebook calls AWS Bedrock API. Configure your AWS credentials:

1. **Option 1 - AWS CLI** (Recommended):
   ```bash
   aws configure
   # Enter your AWS Access Key ID, Secret Access Key, and region (us-east-1)
   ```

2. **Option 2 - Environment Variables**:
   ```bash
   export AWS_ACCESS_KEY_ID=your_key_id
   export AWS_SECRET_ACCESS_KEY=your_secret_key
   export AWS_DEFAULT_REGION=us-region
   ```

3. **Option 3 - .env file** (create in notebook directory):
   ```
   AWS_ACCESS_KEY_ID=your_key_id
   AWS_SECRET_ACCESS_KEY=your_secret_key
   AWS_DEFAULT_REGION=us-east-1
   ```

**Note**: Ensure your AWS user/role has permissions for:
- `bedrock:InvokeModel` or `bedrock:Converse`

In [ ]:
# !pip install pypdfium2 pypdf termcolor faker boto3 python-dotenv pyyaml -q
# Let's make sure that modules are autoreloaded
%load_ext autoreload
%autoreload 2

# First uninstall existing package (to ensure we get the latest version)
%pip uninstall -y pii-anonymizer

# Install the PII anonymizer package in development mode with all extras
%pip install -q -e ".[all]"

# Check installed version
%pip show pii-anonymizer | grep -E "Version|Location"

# Optionally use a .env file for environment variables
try:
    from dotenv import load_dotenv
    load_dotenv()  
except ImportError:
    pass


In [ ]:
### Configure AWS Profile

# **IMPORTANT**: Jupyter doesn't inherit AWS_PROFILE from terminal.

# Set your AWS profile name below (from `aws configure list`):

In [ ]:
# !aws sts get-caller-identity

In [ ]:
# ============================================================
# 1) SET YOUR AWS PROFILE  --  the one thing you must configure
# ============================================================
# This notebook calls AWS (Bedrock, Textract, Transcribe, Polly, S3) and must use
# credentials for the ACCOUNT WHERE YOUR S3 BUCKETS LIVE.
#
# Pick the right profile:
#   1. List profiles:   aws configure list-profiles
#   2. Test one:        aws sts get-caller-identity --profile <name>
#   3. Use the profile whose "Account" matches your buckets' account.
#
# Examples:
#   AWS_PROFILE_NAME = "my-project-Admin"   # a named profile (recommended)
#   AWS_PROFILE_NAME = None                  # use your default profile
#
# If you see "ExpiredToken" later, your login lapsed -> re-authenticate
#   (Midway/Isengard: `mwinit`   |   SSO: `aws sso login --profile <name>`)
# then Kernel -> Restart Kernel and run from the top.

AWS_PROFILE_NAME = None   # <-- CHANGE THIS to your profile name (or leave None)


### Validate AWS Credentials

This cell validates your AWS profile works correctly.

In [ ]:
import boto3
import os

print("Validating AWS credentials...\n")

# Make the chosen profile the default for EVERY boto3 call in this notebook
# (so even plain boto3.client(...) downstream uses it), and drop any session
# cached from a previous (expired) run -- this avoids the classic "ExpiredToken
# even after re-login" trap, which is caused by a stale cached default session.
if AWS_PROFILE_NAME:
    os.environ["AWS_PROFILE"] = AWS_PROFILE_NAME
boto3.DEFAULT_SESSION = None

try:
    if AWS_PROFILE_NAME:
        print(f"Using AWS profile: {AWS_PROFILE_NAME}")
        boto3_session = boto3.Session(profile_name=AWS_PROFILE_NAME)
    else:
        print("Using default credentials")
        boto3_session = boto3.Session()

    # Region comes from the profile (fallback us-east-1). Stored for later cells.
    region = boto3_session.region_name or "us-east-1"
    os.environ["AWS_REGION"] = region

    # Validate against the SELECTED session (not a bare default client)
    identity = boto3_session.client("sts", region_name=region).get_caller_identity()
    account_id = identity["Account"]

    print("\n\u2713 AWS credentials are valid!")
    print(f"  Account:   {account_id}")
    print(f"  User/Role: {identity['Arn']}")
    print(f"  Region:    {region}")

    # Store session for later use
    AWS_SESSION = boto3_session

except Exception as e:
    print("\n\u2717 AWS credential validation failed!")
    print(f"  Error: {e}\n")
    print("Troubleshooting (do these in order):")
    print("  1. Re-authenticate your login:")
    if AWS_PROFILE_NAME:
        print(f"       Midway/Isengard:  mwinit")
        print(f"       SSO:              aws sso login --profile {AWS_PROFILE_NAME}")
    else:
        print("       Midway/Isengard:  mwinit      |   SSO:  aws sso login")
    print("  2. Kernel -> Restart Kernel, then run this notebook from the top.")
    print("     (A kernel restart clears any cached/expired session - this is the")
    print("      usual fix when you still get 'ExpiredToken' right after re-login.)")
    if AWS_PROFILE_NAME:
        print(f"  3. Confirm the profile reaches the right account:")
        print(f"       aws sts get-caller-identity --profile {AWS_PROFILE_NAME}")
    else:
        print("  3. Or set AWS_PROFILE_NAME in the cell above to a working profile.")
    AWS_SESSION = None


In [ ]:
import sys
import os

# Add src directory to Python path
src_path = os.path.join(os.getcwd(), 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    
print(f"Added to path: {src_path}")

In [ ]:
import os, io, json, time, traceback
import logging
import boto3
import math
from botocore.config import Config
from PIL import Image
import matplotlib.pyplot as plt

# Configure logging for the notebook
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


from helpers.model_config_helper import get_inference_config_from_yaml

from helpers.pdf_processor import extract_all_pages_as_images, extract_all_embedded_images
from core.pii_detector import detect_pii_in_image, detect_pii_in_embedded_image
from core.synthetic_pii_generator import batch_generate_synthetic_pii
from redaction.pdf_redactor import redact_pdf
from helpers.text_chunker import chunk_text_by_lines
from helpers.threaded_detector import run_threaded_pii_detection
from core.text_replacer import replace_pii_in_text
from helpers.token_tracker import TokenTracker
from core.prompts import SYSTEM_PROMPT, PII_DETECTION_PROMPT


print("✓ Imports successful")

In [ ]:
# Load Configuration from config.yaml
import yaml

if AWS_SESSION is None:
    print("Cannot proceed - AWS credentials not valid. Please fix credentials and restart kernel.")

    
else:
    config_path = 'src/config.yaml'
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # Extract configuration
    MODEL_ID = config['model']['id']
    MODEL_PROVIDER = config['model']['provider']
    DPI = config['performance']['dpi']
    APPROACH = config['processing']['approach']
    PROCESS_EMBEDDED = config['processing']['process_embedded_images']
    
    # AWS Client Configuration using the validated session
    AWS_CONFIG = Config(
        retries={"max_attempts": config['performance']['max_retries'], "mode": "adaptive"},
        connect_timeout=60,
        read_timeout=config['performance']['timeout_seconds']
    )
    BEDROCK = AWS_SESSION.client('bedrock-runtime', config=AWS_CONFIG)
    
    # Output Configuration
    OUTPUT_DIR = "sample-data/output"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    print("Configuration loaded from config.yaml:")
    print(f"  Model: {MODEL_ID}")
    print(f"  Provider: {MODEL_PROVIDER}")
    print(f"  Approach: {APPROACH}")
    print(f"  DPI: {DPI}")
    print(f"  Process Embedded Images: {PROCESS_EMBEDDED}")
    
    # Toggle PII highlights for Word/Excel output
    config.setdefault('markers', {})['word'] = True
    config.setdefault('markers', {})['tabular'] = True
    print(f"  Highlight Word: {config['markers']['word']}")
    print(f"  Highlight Excel: {config['markers']['tabular']}")
    print(f"  Output: {OUTPUT_DIR}")
    print("\n✓ Ready for testing!")

In [ ]:
# Helper Functions
def compare_images(img1, img2, title1="Original", title2="Redacted"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    ax1.imshow(img1); ax1.set_title(title1, fontsize=14, fontweight='bold'); ax1.axis('off')
    ax2.imshow(img2); ax2.set_title(title2, fontsize=14, fontweight='bold'); ax2.axis('off')
    plt.tight_layout(); plt.show()

def print_summary(title, metrics):
    print(f"\n{'='*60}\n{title}\n{'='*60}")
    for k, v in metrics.items():
        print(f"{k}: {v:.2f}" if isinstance(v, float) else f"{k}: {v}")
    print("=" * 60)

In [ ]:
# Select PDF test file
pdf_files = [f for f in os.listdir('sample-data') if f.lower().endswith('.pdf')]
print(f"Available PDFs ({len(pdf_files)}): {pdf_files}")

test_file = pdf_files[0] if pdf_files else None
# test_file = 'insurance_package_single.pdf'  # Uncomment to pick specific file

if test_file:
    pdf_path = os.path.join('sample-data', test_file)
    filename_without_ext = os.path.splitext(test_file)[0]
    print(f"Testing: {pdf_path}")
else:
    pdf_path = None
    print("No PDFs found in sample-data/")


---
# Section 1: PDF Processing

## Approach A — Text Extraction
Extracts raw text from PDF, best for text-heavy documents. Output: `.txt`

- **Step 1 — Detection**: Extract text → chunk → threaded PII detection via Bedrock
- **Step 2 — Synthetic**: Batch generate synthetic replacements via LLM
- **Step 3 — Redaction**: Replace PII in text using longest-first ordering


In [ ]:
if pdf_path:
    print("\n" + "="*60 + "\nTEXT-BASED APPROACH\n" + "="*60)
    
    try:
        start = time.time()
        from pypdf import PdfReader
        from helpers.model_config_helper import get_concurrency_config
        
        # Step 1: Extract text from PDF
        reader = PdfReader(pdf_path)
        full_text = "\n\n".join(page.extract_text() or "" for page in reader.pages)
        print(f"Extracted {len(full_text)} chars from {len(reader.pages)} pages")
        
        # Step 2: Chunk and detect PII (threaded)
        cc = get_concurrency_config(config)
        chunks_list = chunk_text_by_lines(full_text, cc['max_txt_chunk_tokens'], cc['chars_per_token'])
        chunks = {f"Chunk {i+1}": c for i, c in enumerate(chunks_list)}
        inference_config = get_inference_config_from_yaml(config)
        tracker = TokenTracker(MODEL_ID)
        all_pii, failed, _ = run_threaded_pii_detection(
            chunks, MODEL_ID, MODEL_PROVIDER, BEDROCK,
            inference_config, cc['max_workers'], token_tracker=tracker
        )
        print(f"Detected {len(all_pii)} PII entities")
        
        # Step 3: Generate synthetic replacements
        synthetic_map = batch_generate_synthetic_pii(
            all_pii, MODEL_ID, MODEL_PROVIDER, BEDROCK,
            config=config, token_tracker=tracker
        )
        
        # Step 4: Replace PII in text
        redacted_text, _, _, _, _ = replace_pii_in_text(full_text, synthetic_map)
        
        text_time = time.time() - start
        output_path = os.path.join(OUTPUT_DIR, f"text_{filename_without_ext}.txt")
        with open(output_path, 'w') as f:
            f.write(redacted_text)
        
        print_summary("TEXT-BASED RESULTS", {
            "Time (s)": text_time,
            "PII Count": len(all_pii),
            "Output": output_path
        })
        text_metrics = {"time": text_time, "pii": len(all_pii)}
        
    except Exception as e:
        print(f"\n✗ Text-based approach failed")
        print(f"   Error: {str(e)[:200]}")
        traceback.print_exc()
        text_metrics = None
else:
    text_metrics = None


## Approach B — Image-Based (Vision LLM)
Converts PDF pages to images, best for scanned documents and complex layouts. Output: `.pdf`

- **Step 1 — Detection**: Threaded page processing → Vision LLM + Textract bounding box refinement
- **Step 2 — Synthetic**: Generate synthetic replacements via LLM (Faker fallback)
- **Step 3 — Redaction**: flatten-to-image pixel redaction on the PDF

### Redaction Options (`src/config.yaml` → `redaction:`)

| Option | Values | Description |
|--------|--------|-------------|
| `mode` | `synthetic` (default) / `blackout` | Synthetic replaces with fake data; blackout fills with black rectangles and skips LLM synthetic generation |
| `show_bounding_boxes` | `true` / `false` | Draws colored outlines around redacted areas for visual verification |
| `bounding_box_color` | `[R, G, B]` (0-1 floats) | Box outline color, default `[1, 0, 0]` (red) |


In [ ]:
if pdf_path:
    logger.info("\n" + "="*60 + "\nIMAGE-BASED: Detection & Redaction\n" + "="*60)
    
    from helpers.page_type_checker import get_text_based_pages
    from helpers.model_config_helper import get_concurrency_config
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from helpers.textract_helper import enhance_pii_detections_with_textract, get_textract_full
    
    text_pages = get_text_based_pages(pdf_path, text_threshold=50)
    logger.info(f"Found {len(text_pages)} text-based pages: {text_pages}")
    
    page_images = extract_all_pages_as_images(pdf_path)
    logger.info(f"✓ {len(page_images)} pages extracted")
    
    if page_images:
        plt.figure(figsize=(10, 10))
        plt.imshow(page_images[0][0])
        plt.title("Original Page 1", fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.show()
    
    # Step 1: Threaded PII detection with Textract bounding box refinement
    # Page screenshots capture embedded images — no separate embedded image processing needed
    cc = get_concurrency_config(config)
    max_workers = cc['max_workers']
    tracker = TokenTracker(MODEL_ID)
    detection_start = time.time()
    completed = [0]
    
    def _detect_page(img, metadata, page_num):
        textract_words, ocr_text, _raw = get_textract_full(img, BEDROCK)
        detections = detect_pii_in_image(img, metadata, MODEL_ID, MODEL_PROVIDER, BEDROCK, token_tracker=tracker, ocr_text=ocr_text)
        if detections:
            detections = enhance_pii_detections_with_textract(img, detections, BEDROCK, textract_words=textract_words)
        completed[0] += 1
        logger.info(f"  [{completed[0]}/{len(page_work)}] Page {page_num}: {len(detections)} detections")
        return detections
    
    page_work = [(img, meta, meta.get('page_number', i)) for i, (img, meta) in enumerate(page_images)]
    logger.info(f"Step 1 - PII Detection: {len(page_work)} pages with {max_workers} workers")
    
    all_detections = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_detect_page, img, meta, pn): pn for img, meta, pn in page_work}
        for future in as_completed(futures):
            try:
                all_detections.extend(future.result())
            except Exception as e:
                logger.error(f"Page {futures[future]} failed: {e}")
    
    detection_time = time.time() - detection_start
    redactable = [d for d in all_detections if d.get('bounding_box')]
    skipped = [d for d in all_detections if not d.get('bounding_box')]
    logger.info(f"✓ {len(all_detections)} PII ({len(redactable)} with bbox, {len(skipped)} skipped) in {detection_time:.1f}s")
    
    # Step 2: Generate synthetic replacements (or blackout)
    redaction_mode = config.get('redaction', {}).get('mode', 'synthetic')
    if redaction_mode == 'blackout':
        logger.info("Blackout mode — skipping synthetic generation")
        pii_mapping = {d.get('content', ''): '[REDACTED]' for d in redactable}
    else:
        pii_mapping = batch_generate_synthetic_pii(redactable, MODEL_ID, MODEL_PROVIDER, BEDROCK, config=config, token_tracker=tracker)
        logger.info(f"✓ Generated {len(pii_mapping)} synthetic replacements")
    
    detection_metrics = {'time': detection_time, 'pii': len(all_detections), 'redactable': len(redactable), 'skipped': len(skipped)}
else:
    all_detections = []
    pii_mapping = {}
    detection_metrics = None


In [ ]:
# Visualize embedded images (debug/exploration — not used in detection pipeline)
embedded_images = extract_all_embedded_images(pdf_path) if pdf_path else []
if embedded_images:
    cols = min(4, len(embedded_images))
    rows = math.ceil(len(embedded_images) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
    axes = [axes] if len(embedded_images) == 1 else axes.flatten()
    for i, (img, meta) in enumerate(embedded_images):
        axes[i].imshow(img)
        axes[i].set_title(f"Embedded {i+1} (p{meta['page_number']})")
        axes[i].axis('off')
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No embedded images found')


In [ ]:
page_images

In [ ]:
embedded_images

In [ ]:
all_detections

In [ ]:
pii_mapping

## Step 3 — PDF Redaction

Applies redaction to the PDF using the flatten-to-image approach (each page rendered and pixel-redacted, no text layer).

### Redaction Options (`src/config.yaml` → `redaction:`)

| Option | Values | Description |
|--------|--------|-------------|
| `mode` | `synthetic` (default) / `blackout` | Synthetic replaces PII with fake data; blackout fills with black rectangles |
| `show_bounding_boxes` | `true` / `false` | Draws colored outlines around redacted areas for visual verification |
| `bounding_box_color` | `[R, G, B]` (0-1 floats) | Box outline color, default `[1, 0, 0]` (red) |

The `dpi` setting from `performance:` controls render resolution for image-based pages.


In [ ]:
from helpers.model_config_helper import get_show_bounding_boxes
if pdf_path and all_detections:
    print("\n" + "="*60 + "\nSTEP 3 — PDF REDACTION\n" + "="*60)
    
    file_path = os.path.join(OUTPUT_DIR, f"redacted_{filename_without_ext}.pdf")
    summary_path = os.path.join(OUTPUT_DIR, f"summary_{filename_without_ext}.json")
    
    # Match Lambda: pass redaction config from config.yaml
    redaction_config = {**config.get('redaction', {}), 'show_bounding_boxes': get_show_bounding_boxes(config), 'dpi': config.get('performance', {}).get('dpi', 300)}
    
    file_start = time.time()
    success = redact_pdf(
        pdf_path, file_path, all_detections, pii_mapping, summary_path, None,
        token_usage=tracker.summary(), bedrock_runtime=BEDROCK, redaction_config=redaction_config
    )
    file_time = time.time() - file_start
    
    if success:
        print(f"✓ Redacted PDF: {file_path} ({file_time:.1f}s)")
        print(f"  Mode: {redaction_config.get('mode', 'synthetic')}, Bounding boxes: {redaction_config.get('show_bounding_boxes', False)}")
        
        file_pages = extract_all_pages_as_images(file_path)
        if file_pages and page_images:
            compare_images(page_images[0][0], file_pages[0][0], "Original", "Redacted")
        
        file_metrics = {'time': file_time, 'success': True}
    else:
        print("✗ Redaction failed")
        file_metrics = {'time': file_time, 'success': False}
else:
    file_metrics = None


---
# Section 2: Document Processing

## Word (.docx)
Same 3-step pipeline — extracts text per page, detects PII, replaces across paragraphs and tables.

- **Step 1 — Detection**: Extract text per page → threaded PII detection
- **Step 2 — Synthetic**: Batch generate replacements via LLM
- **Step 3 — Redaction**: Replace PII in paragraphs and tables


In [ ]:
# Select Word test file
docx_files = [f for f in os.listdir('sample-data') if f.lower().endswith('.docx')]
print(f"Available Word files ({len(docx_files)}): {docx_files}")

docx_file = docx_files[0] if docx_files else None
# docx_file = 'my_document.docx'  # Uncomment to pick specific file

if docx_file:
    docx_path = os.path.join('sample-data', docx_file)
    print(f"Testing: {docx_path}")
else:
    docx_path = None
    print("No .docx files found in sample-data/")


In [ ]:
if docx_path:
    print("\n" + "="*60 + "\nWORD PROCESSING\n" + "="*60)
    
    try:
        start = time.time()
        from docx import Document
        from processors.word_processor import _build_page_chunks, replace_pii_in_word
        from helpers.model_config_helper import get_concurrency_config
        
        # Step 1: Extract text and detect PII
        doc = Document(docx_path)
        chunks = _build_page_chunks(doc)
        cc = get_concurrency_config(config)
        inference_config = get_inference_config_from_yaml(config)
        tracker = TokenTracker(MODEL_ID)
        all_pii, failed, _ = run_threaded_pii_detection(
            chunks, MODEL_ID, MODEL_PROVIDER, BEDROCK,
            inference_config, cc['max_workers'], label='Page', token_tracker=tracker
        )
        print(f"Detected {len(all_pii)} PII entities")
        
        # Step 2: Generate synthetic replacements
        synthetic_map = batch_generate_synthetic_pii(
            all_pii, MODEL_ID, MODEL_PROVIDER, BEDROCK,
            config=config, token_tracker=tracker
        )
        
        # Step 3: Replace PII in Word document
        output_path = os.path.join(OUTPUT_DIR, f"redacted_{filename_without_ext}.docx")
        word_result = replace_pii_in_word(docx_path, output_path, synthetic_map)
        
        word_time = time.time() - start
        print_summary("WORD RESULTS", {
            "Time (s)": word_time,
            "PII Detected": len(all_pii),
            "Replacements": word_result.get("replacements", 0),
            "Output": output_path
        })
        
    except Exception as e:
        print(f"\n✗ Word processing failed: {e}")
        traceback.print_exc()


## Excel (.xlsx)
Same 3-step pipeline — extracts text from all sheets, detects PII, replaces cell-by-cell.

- **Step 1 — Detection**: Extract text from all sheets → threaded PII detection
- **Step 2 — Synthetic**: Batch generate replacements via LLM
- **Step 3 — Redaction**: Replace PII cell-by-cell across all sheets


In [ ]:
# Select Excel test file
xlsx_files = [f for f in os.listdir('sample-data') if f.lower().endswith('.xlsx')]
print(f"Available Excel files ({len(xlsx_files)}): {xlsx_files}")

xlsx_file = xlsx_files[0] if xlsx_files else None
# xlsx_file = 'my_spreadsheet.xlsx'  # Uncomment to pick specific file

if xlsx_file:
    xlsx_path = os.path.join('sample-data', xlsx_file)
    print(f"Testing: {xlsx_path}")
else:
    xlsx_path = None
    print("No .xlsx files found in sample-data/")


In [ ]:
if xlsx_path:
    print("\n" + "="*60 + "\nEXCEL PROCESSING\n" + "="*60)
    
    try:
        start = time.time()
        from processors.tabular_processor import extract_text_from_excel, replace_pii_in_excel
        from helpers.model_config_helper import get_concurrency_config
        
        # Step 1: Extract text and detect PII
        result = extract_text_from_excel(xlsx_path)
        sheets_data, all_text = result['sheets'], result['all_text']
        cc = get_concurrency_config(config)
        chunks = {s['sheet_name']: ' '.join(c['value'] for c in s['cells']) for s in sheets_data}
        inference_config = get_inference_config_from_yaml(config)
        tracker = TokenTracker(MODEL_ID)
        all_pii, failed, _ = run_threaded_pii_detection(
            chunks, MODEL_ID, MODEL_PROVIDER, BEDROCK,
            inference_config, cc['max_workers'], label='Sheet', token_tracker=tracker
        )
        print(f"Detected {len(all_pii)} PII entities")
        
        # Step 2: Generate synthetic replacements
        synthetic_map = batch_generate_synthetic_pii(
            all_pii, MODEL_ID, MODEL_PROVIDER, BEDROCK,
            config=config, token_tracker=tracker
        )
        
        # Step 3: Replace PII in Excel
        output_xlsx = os.path.join(OUTPUT_DIR, f"redacted_{os.path.splitext(os.path.basename(xlsx_path))[0]}.xlsx")
        excel_result = replace_pii_in_excel(xlsx_path, output_xlsx, synthetic_map)
        
        excel_time = time.time() - start
        print_summary("EXCEL RESULTS", {
            "Time (s)": excel_time,
            "Sheets": result['total_sheets'],
            "PII Detected": len(all_pii),
            "Replacements": excel_result.get("replacements", 0),
            "Output": output_xlsx
        })
        
    except Exception as e:
        print(f"\n✗ Excel processing failed: {e}")
        traceback.print_exc()


## Plain Text (.txt)
Simplest pipeline — reads text, detects PII, replaces inline.

- **Step 1 — Detection**: Read text → chunk → threaded PII detection
- **Step 2 — Synthetic**: Generate synthetic replacements
- **Step 3 — Redaction**: Replace PII in text and write output

In [ ]:
# Select TXT test file
txt_files = [f for f in os.listdir('sample-data') if f.lower().endswith('.txt')]
print(f"Available TXT files: {txt_files}")
txt_path = os.path.join('sample-data', txt_files[0]) if txt_files else None
if txt_path:
    print(f"Selected: {txt_path}")

In [ ]:
if txt_path:
    print("\n" + "="*60 + "\nTXT PROCESSING\n" + "="*60)

    try:
        start = time.time()
        from helpers.model_config_helper import get_concurrency_config
        from helpers.text_chunker import chunk_text_by_lines

        with open(txt_path, 'r') as f:
            text = f.read()
        print(f"Read {len(text)} chars from {txt_path}")

        # Step 1 — Detection
        inference_config = get_inference_config_from_yaml(config)
        cc = get_concurrency_config(config)
        max_tokens = config.get('processing', {}).get('max_chunk_tokens', 1000)
        chars_per_token = config.get('processing', {}).get('chars_per_token', 3)
        chunks = chunk_text_by_lines(text, max_tokens, chars_per_token)
        chunks = {f'Chunk {j+1}': c for j, c in enumerate(chunks)}
        print(f"Split into {len(chunks)} chunks")

        tracker = TokenTracker(MODEL_ID)
        all_detections, failed, _ = run_threaded_pii_detection(
            chunks, MODEL_ID, MODEL_PROVIDER, BEDROCK,
            inference_config, cc['max_workers'], label='Chunk', token_tracker=tracker
        )
        print(f"Detected {len(all_detections)} PII items")

        # Step 2 — Synthetic
        synthetic_map = batch_generate_synthetic_pii(
            all_detections, MODEL_ID, MODEL_PROVIDER, BEDROCK,
            config=config, token_tracker=tracker
        )
        print(f"Generated {len(synthetic_map)} synthetic replacements")

        # Step 3 — Redaction
        redacted_text, _, _, _, _ = replace_pii_in_text(text, synthetic_map)

        out_path = os.path.join(OUTPUT_DIR, f"redacted_{os.path.basename(txt_path)}")
        with open(out_path, 'w') as f:
            f.write(redacted_text)

        elapsed = time.time() - start
        print(f"\n✅ TXT redaction complete in {elapsed:.1f}s → {out_path}")
        print(f"\nSample (first 500 chars):\n{redacted_text[:500]}")

    except Exception as e:
        logger.error(f"TXT processing failed: {e}")
        traceback.print_exc()

## Images (.jpg, .png, .tiff, etc.)
Detects PII directly from standalone images using Vision LLM.

- **Step 1 — Detection**: Vision LLM detects PII from image
- **Step 2 — Synthetic**: Generate synthetic replacements via LLM
- **Step 3 — Output**: Display detected PII and mappings (no in-place image redaction)


In [ ]:
# Select image test file
IMAGE_EXT = ('.jpg', '.jpeg', '.png', '.tiff', '.tif', '.bmp', '.webp')
img_files = [f for f in os.listdir('sample-data') if f.lower().endswith(IMAGE_EXT)]
print(f"Available images ({len(img_files)}): {img_files}")

img_file = img_files[0] if img_files else None
# img_file = 'my_image.png'  # Uncomment to pick specific file

if img_file:
    img_path = os.path.join('sample-data', img_file)
    print(f"Testing: {img_path}")
else:
    img_path = None
    print("No image files found in sample-data/")


In [ ]:
if img_path:
    print("\n" + "="*60 + "\nIMAGE PROCESSING\n" + "="*60)
    
    try:
        from PIL import Image, ImageDraw
        from helpers.textract_helper import enhance_pii_detections_with_textract, get_textract_full
        from processors.image_processor import draw_synthetic_text_on_image
        start = time.time()
        
        img = Image.open(img_path)
        metadata = {'page_number': 1, 'width': img.width, 'height': img.height,
                    'page_width': img.width, 'page_height': img.height, 'scale_x': 1.0, 'scale_y': 1.0}
        
        plt.figure(figsize=(10, 10))
        plt.imshow(img)
        plt.title('Original Image', fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.show()
        
        # Step 1: Detect PII with Textract OCR context + bounding box refinement
        tracker = TokenTracker(MODEL_ID)
        textract_words, ocr_text, _raw = get_textract_full(img, BEDROCK)
        detections = detect_pii_in_image(img, metadata, MODEL_ID, MODEL_PROVIDER, BEDROCK, token_tracker=tracker, ocr_text=ocr_text)
        if detections:
            detections = enhance_pii_detections_with_textract(img, detections, BEDROCK, textract_words=textract_words)
        print(f"\n✓ Detected {len(detections)} PII entities in {time.time()-start:.1f}s")
        
        # Step 2: Generate synthetic replacements (or blackout)
        redaction_mode = config.get('redaction', {}).get('mode', 'synthetic')
        from helpers.model_config_helper import get_show_bounding_boxes
        show_boxes = get_show_bounding_boxes(config)
        box_color = tuple(config.get('redaction', {}).get('bounding_box_color', [255, 0, 0]))
        
        if detections:
            redactable = [d for d in detections if d.get('bounding_box')]
            if redaction_mode == 'blackout':
                pii_mapping = {d.get('content', ''): '[REDACTED]' for d in redactable}
            else:
                pii_mapping = batch_generate_synthetic_pii(redactable, MODEL_ID, MODEL_PROVIDER, BEDROCK, config=config, token_tracker=tracker)
            print(f"✓ {len(pii_mapping)} mappings (mode: {redaction_mode})")
        else:
            redactable, pii_mapping = [], {}
        
        # Step 3: Redact image
        img_redacted = img.copy()
        draw = ImageDraw.Draw(img_redacted)
        for det in redactable:
            bbox = det.get('bounding_box')
            if not bbox:
                continue
            left, top = int(bbox['left']), int(bbox['top'])
            right, bottom = int(bbox['left'] + bbox['width']), int(bbox['top'] + bbox['height'])
            if redaction_mode == 'blackout':
                draw.rectangle([left, top, right, bottom], fill=(0, 0, 0))
            else:
                draw.rectangle([left, top, right, bottom], fill=(255, 255, 255))
                draw_synthetic_text_on_image(draw, pii_mapping.get(det['content'], ''), left, top, right-left, bottom-top)
            if show_boxes:
                draw.rectangle([left, top, right, bottom], outline=box_color, width=3)
        
        # Save and display
        img_name = os.path.splitext(os.path.basename(img_path))[0]
        img_ext = os.path.splitext(img_path)[1].lower()
        output_img = os.path.join(OUTPUT_DIR, f"redacted_{img_name}{img_ext}")
        save_kwargs = {'compression': 'tiff_deflate'} if img_ext in ['.tiff', '.tif'] else {}
        img_redacted.save(output_img, **save_kwargs)
        print(f"✓ Redacted image: {output_img}")
        
        compare_images(img, img_redacted, 'Original', 'Redacted')
        
    except Exception as e:
        print(f"\n✗ Image processing failed: {e}")
        traceback.print_exc()


# Section 3: Audio Processing

Audio redaction works differently from the document formats above: it relies on **AWS services end to end** rather than purely local processing.

1. **Amazon Transcribe** transcribes the audio from S3 (word-level timestamps + speaker diarization).
2. A **Bedrock** model detects PII in the transcript text.
3. Synthetic replacements are generated (or silence, per `audio.redaction_mode`).
4. **Amazon Polly** synthesizes replacement speech and **ffmpeg** splices it at the exact PII timestamps.

> Requires deployed **input/output S3 buckets**, IAM access to Transcribe/Polly, and **ffmpeg** available locally (`brew install ffmpeg`). Set `INPUT_BUCKET`/`OUTPUT_BUCKET` below to your deployed buckets.

In [ ]:
import os, shutil
from core.synthetic_pii_generator import batch_generate_synthetic_pii

# Audio splicing uses ffmpeg. The pipeline resolves it via shutil.which("ffmpeg")
# at import time, falling back to the Lambda path /opt/bin/ffmpeg. Locally, make
# sure ffmpeg's directory is on PATH so it is found (install: brew install ffmpeg).
for _d in ["/opt/homebrew/bin", "/usr/local/bin", "/usr/bin"]:
    if os.path.isdir(_d) and _d not in os.environ.get("PATH", ""):
        os.environ["PATH"] = _d + os.pathsep + os.environ.get("PATH", "")
assert shutil.which("ffmpeg"), "ffmpeg not found - run: brew install ffmpeg, then restart the kernel"

from processors import audio_processor
from processors.audio_processor import detect_pii_audio, redact_audio
audio_processor.FFMPEG = shutil.which("ffmpeg")  # ensure the local ffmpeg path is used

# Audio uses S3 + Transcribe + Polly + ffmpeg (not purely local). Point these
# at your deployed buckets:
INPUT_BUCKET = "my-pii-input1"     # your deployed input bucket
OUTPUT_BUCKET = "my-pii-output1"   # your deployed output bucket

s3_client = AWS_SESSION.client("s3", config=AWS_CONFIG)

# Transcribe must run in the SAME region as the bucket (else 'incorrect S3 URI').
bucket_region = s3_client.get_bucket_location(Bucket=INPUT_BUCKET)["LocationConstraint"] or "us-east-1"
os.environ["AWS_REGION"] = bucket_region
os.environ["AWS_DEFAULT_REGION"] = bucket_region
print(f"Using region {bucket_region} for Transcribe/Polly (matches bucket)")

audio_file = "sample-data/audio/dialog.mp3"
audio_key = f"notebook-audio/{os.path.basename(audio_file)}"
s3_client.upload_file(audio_file, INPUT_BUCKET, audio_key)
print(f"Uploaded {audio_file} -> s3://{INPUT_BUCKET}/{audio_key}")

# Step 1 - Transcribe + detect PII in the transcript (async Transcribe job; can take 1-2 min)
detection_data = detect_pii_audio(INPUT_BUCKET, audio_key, config, BEDROCK, s3_client)
detections = detection_data["detections"]
print(f"\nTranscript (first 200 chars):\n{detection_data['transcript'][:200]}...")
print(f"\nDetected {len(detections)} PII entities:")
for d in detections[:10]:
    print(f"  - {d.get('type','?')}: {d.get('content','')}")

# Step 2 - synthetic replacements (redact_audio honors audio.redaction_mode: synthetic|silence)
pii_mapping = batch_generate_synthetic_pii(detections, MODEL_ID, MODEL_PROVIDER, BEDROCK, config=config)

# Step 3 - synthesize/splice -> anonymized WAV + redacted transcript in S3
output_key, replaced, found = redact_audio(
    s3_client, INPUT_BUCKET, audio_key, OUTPUT_BUCKET,
    pii_mapping, detections, detection_data, config,
    job_id="notebook-audio", output_prefix="notebook-audio",
)
print(f"\nReplaced {replaced} PII occurrence(s). Output: s3://{OUTPUT_BUCKET}/{output_key}")

# Download + play the redacted audio
os.makedirs(OUTPUT_DIR, exist_ok=True)
local_out = os.path.join(OUTPUT_DIR, os.path.basename(output_key))
s3_client.download_file(OUTPUT_BUCKET, output_key, local_out)
from IPython.display import Audio
Audio(local_out)